### Partition Pruning
* Partition pruning is a query optimization technique that **skips irrelevant partitions** or folders of a table during a query, reducing disk I/O and speeding up execution.

#### How It Works
* **Data Organization:** Tables are split into partitions based on specific keys (like date or region). 
* **Filter Evaluation:** The database optimizer checks query conditions ( clauses or joins) against partition metadata. 
* **Exclusion:** Any partition that cannot possibly contain matching data is completely ignored, meaning the system reads only the required slices of data.

#### Types of Partition Pruning 
* **Static Partition Pruning:** Occurs at compile time when filter values are explicit constants known before execution (e.g., ). 
* **Dynamic Partition Pruning (DPP):** Occurs at runtime when filter values depend on the results of another table or a join condition, allowing the engine to prune partitions on the fly after evaluating the first dataset.


#### Below Example explain the dynamic partition pruning

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import date_format, col

In [2]:
spark = SparkSession.Builder().appName("songDataset").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/21 01:34:26 WARN Utils: Your hostname, Vishvas-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.79 instead (on interface en0)
26/09/21 01:34:26 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/21 01:34:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
song_df = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("data/Spotify_Songs.csv")

# song_df.show(5, False)

# updating the timestamp to date
song_df=song_df.withColumnRenamed("release_date", "release_ts")\
    .withColumn("release_date", date_format(col("release_ts"), "yyyy-MM-dd"))
song_df.show(5, False)

+-------+------+---------+--------------------------+------------+
|song_id|title |artist_id|release_ts                |release_date|
+-------+------+---------+--------------------------+------------+
|1      |Song_1|2        |2021-10-15 10:15:47.006571|2021-10-15  |
|2      |Song_2|45       |2020-12-07 10:15:47.006588|2020-12-07  |
|3      |Song_3|25       |2022-07-11 10:15:47.006591|2022-07-11  |
|4      |Song_4|25       |2019-03-09 10:15:47.006593|2019-03-09  |
|5      |Song_5|26       |2019-09-07 10:15:47.006596|2019-09-07  |
+-------+------+---------+--------------------------+------------+
only showing top 5 rows


In [4]:
# Reading data from csv file
listening_df_csv = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", True)\
    .load("data/Spotify_Listening_activity.csv")

# udpating the timestamp column
listening_df_csv= listening_df_csv.withColumnRenamed("listen_date", "listen_ts")\
    .withColumn("listen_date", date_format(col("listen_ts"), "yyyy-MM-dd"))
# listening_df_csv.show(5, False)

# saving as parquet to enable the partition on listen_date column
listening_df_csv.write\
    .mode("overwrite")\
    .partitionBy("listen_date")\
    .parquet("data/Spotify_listening_df")

# reading from parquet file
listening_df = spark.read.parquet("data/Spotify_listening_df")
listening_df.show(5, False)

+-----------+-------+--------------------------+---------------+-----------+
|activity_id|song_id|listen_ts                 |listen_duration|listen_date|
+-----------+-------+--------------------------+---------------+-----------+
|4456       |16     |2023-07-18 10:15:47.023264|151            |2023-07-18 |
|4457       |65     |2023-07-18 10:15:47.023264|181            |2023-07-18 |
|4458       |60     |2023-07-18 10:15:47.023264|280            |2023-07-18 |
|4459       |3      |2023-07-18 10:15:47.023264|249            |2023-07-18 |
|4460       |45     |2023-07-18 10:15:47.023264|130            |2023-07-18 |
+-----------+-------+--------------------------+---------------+-----------+
only showing top 5 rows


#### Problem Statement
* Analyze listening activity of user:
    1. After 2019-12-31
    2. On release date song


In [5]:
selected_song_df = song_df.filter(col("release_date")>'2019-12-31')
selected_song_df.show(5, False)

+-------+------+---------+--------------------------+------------+
|song_id|title |artist_id|release_ts                |release_date|
+-------+------+---------+--------------------------+------------+
|1      |Song_1|2        |2021-10-15 10:15:47.006571|2021-10-15  |
|2      |Song_2|45       |2020-12-07 10:15:47.006588|2020-12-07  |
|3      |Song_3|25       |2022-07-11 10:15:47.006591|2022-07-11  |
|6      |Song_6|27       |2023-03-25 10:15:47.006598|2023-03-25  |
|7      |Song_7|34       |2023-01-07 10:15:47.006602|2023-01-07  |
+-------+------+---------+--------------------------+------------+
only showing top 5 rows


In [6]:
# joining
result = selected_song_df\
    .join(listening_df,
    on=(selected_song_df.release_date==listening_df.listen_date) &
    (selected_song_df.song_id==listening_df.song_id),
    how="inner")

In [7]:
result.show(10000, False)

+-------+-------+---------+--------------------------+------------+-----------+-------+--------------------------+---------------+-----------+
|song_id|title  |artist_id|release_ts                |release_date|activity_id|song_id|listen_ts                 |listen_duration|listen_date|
+-------+-------+---------+--------------------------+------------+-----------+-------+--------------------------+---------------+-----------+
|89     |Song_89|33       |2023-07-24 10:15:47.006798|2023-07-24  |9760       |89     |2023-07-24 10:15:47.038094|81             |2023-07-24 |
|89     |Song_89|33       |2023-07-24 10:15:47.006798|2023-07-24  |9768       |89     |2023-07-24 10:15:47.038094|295            |2023-07-24 |
|89     |Song_89|33       |2023-07-24 10:15:47.006798|2023-07-24  |9799       |89     |2023-07-24 10:15:47.038094|272            |2023-07-24 |
|64     |Song_64|32       |2023-10-25 10:15:47.006739|2023-10-25  |7322       |64     |2023-10-25 10:15:47.031808|95             |2023-10-25 |

In [8]:
spark